# Hillenbrand & McCarthy (2026) — Monte Carlo Simulation

Monte Carlo simulation engine for the trend-cycle present-value model.

**State vector:** $s_t = (\tau_t, g_t, c_t, \varepsilon^c_t, \mu_t, 1)^\top$  
**Observables:** $x_t = (d_t, pd_t)^\top$

## Setup

In [1]:
include(joinpath(@__DIR__, "HillenbrandMcCarthyModel.jl"))
using .HillenbrandMcCarthyModel
using Random, Statistics

## Configuration

In [2]:
N_PATHS      = 10_000
T_QUARTERS   = 100        # 25 years
BASE_SEED    = 20260316   # paper date as seed
HORIZONS_YRS = [1, 5, 10, 25]
HORIZONS_Q   = HORIZONS_YRS .* 4;

## Data Structures

In [3]:
"""
    HorizonSnapshot

Collected values across all Monte Carlo paths at a single horizon.
"""
struct HorizonSnapshot
    horizon_years::Int
    d::Vector{Float64}          # log dividends
    g::Vector{Float64}          # trend growth (quarterly)
    c::Vector{Float64}          # cycle
    μ::Vector{Float64}          # expected return (quarterly)
    pd::Vector{Float64}         # log price-dividend ratio
    cum_log_r::Vector{Float64}  # cumulative log return from t=0
    ann_log_r::Vector{Float64}  # annualised log return
end

"""
    MCResults

Full Monte Carlo output.
"""
struct MCResults
    params::ModelParams
    n_paths::Int
    t_quarters::Int
    snapshots::Dict{Int, HorizonSnapshot}  # keyed by horizon in years
    sample_paths::Vector{SimulationResult} # 3 full paths for plotting
end

MCResults

## Monte Carlo Simulation

In [4]:
function run_monte_carlo(; n_paths=N_PATHS, t_quarters=T_QUARTERS,
                           horizons_q=HORIZONS_Q, horizons_yrs=HORIZONS_YRS)

    p = default_params()

    println("═══════════════════════════════════════════════════")
    println("  Hillenbrand & McCarthy (2026) Monte Carlo")
    println("═══════════════════════════════════════════════════")
    println("  Paths:      $n_paths")
    println("  Quarters:   $t_quarters  ($(t_quarters÷4) years)")
    println("  Horizons:   $horizons_yrs years")
    println("  Parameters:")
    println("    ρ_c   = $(p.ρ_c)")
    println("    θ_c   = $(p.θ_c)")
    println("    σ_g   = $(p.σ_g)  ($(round(p.σ_g*100*4*sqrt(4), digits=2)) bps ann.)")
    println("    σ_c   = $(p.σ_c)")
    println("    ρ_μ   = $(p.ρ_μ)  (annual: $(round(p.ρ_μ^4, digits=4)))")
    println("    σ_μ   = $(p.σ_μ)")
    println("    μ̄     = $(p.μ_bar)  ($(round(p.μ_bar*4*100, digits=2))% ann.)")
    println("═══════════════════════════════════════════════════")

    # Only collect horizons that fit within simulation length
    valid_mask = horizons_q .<= t_quarters
    valid_h_q  = horizons_q[valid_mask]
    valid_h_y  = horizons_yrs[valid_mask]

    collectors = Dict{Int, HorizonSnapshot}()
    for (hq, hy) in zip(valid_h_q, valid_h_y)
        collectors[hy] = HorizonSnapshot(
            hy,
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths),
            Vector{Float64}(undef, n_paths)
        )
    end

    # Select 3 sample paths to store fully (for plotting)
    sample_indices = [1, div(n_paths, 2), n_paths]
    sample_paths = Vector{SimulationResult}(undef, 3)

    print("  Simulating...")
    t_start = time()

    for i in 1:n_paths
        sim = simulate_path(p, t_quarters; seed=BASE_SEED + i)

        for (hq, hy) in zip(valid_h_q, valid_h_y)
            vals = extract_at_horizon(sim, hq)
            snap = collectors[hy]
            snap.d[i]         = vals.d
            snap.g[i]         = vals.g
            snap.c[i]         = vals.c
            snap.μ[i]         = vals.μ
            snap.pd[i]        = vals.pd
            snap.cum_log_r[i] = vals.cum_log_r
            snap.ann_log_r[i] = annualised_cumulative_return(vals.cum_log_r, Float64(hy))
        end

        idx_in_samples = findfirst(==(i), sample_indices)
        if !isnothing(idx_in_samples)
            sample_paths[idx_in_samples] = sim
        end
    end

    elapsed = round(time() - t_start, digits=2)
    println(" done in $(elapsed)s")

    return MCResults(p, n_paths, t_quarters, collectors, sample_paths)
end

run_monte_carlo (generic function with 1 method)

## Save Results to CSV

In [5]:
function save_results(results::MCResults, output_dir::String)
    mkpath(output_dir)

    for (hy, snap) in results.snapshots
        fname = joinpath(output_dir, "horizon_$(hy)yr.csv")
        open(fname, "w") do io
            println(io, "d,g,c,mu,pd,cum_log_r,ann_log_r")
            for i in 1:results.n_paths
                println(io, "$(snap.d[i]),$(snap.g[i]),$(snap.c[i]),$(snap.μ[i]),$(snap.pd[i]),$(snap.cum_log_r[i]),$(snap.ann_log_r[i])")
            end
        end
        println("  Saved: $fname")
    end

    for (k, sim) in enumerate(results.sample_paths)
        fname = joinpath(output_dir, "sample_path_$(k).csv")
        n = sim.T + 1
        open(fname, "w") do io
            println(io, "quarter,d,tau,g,c,mu,pd,cum_log_r")
            for t in 1:n
                q = t - 1
                println(io, "$q,$(sim.d[t]),$(sim.τ[t]),$(sim.g[t]),$(sim.c[t]),$(sim.μ[t]),$(sim.pd[t]),$(sim.cum_log_r[t])")
            end
        end
        println("  Saved: $fname")
    end

    println("  All results saved to: $output_dir")
end

save_results (generic function with 1 method)

## Run Simulation

In [6]:
results = run_monte_carlo()

#= output_dir = joinpath(@__DIR__, "results")
println("\nSaving results...")
save_results(results, output_dir) =#

═══════════════════════════════════════════════════
  Hillenbrand & McCarthy (2026) Monte Carlo
═══════════════════════════════════════════════════
  Paths:      10000
  Quarters:   100  (25 years)
  Horizons:   [1, 5, 10, 25] years
  Parameters:
    ρ_c   = 0.971
    θ_c   = 0.4927
    σ_g   = 0.000226  (0.18 bps ann.)
    σ_c   = 0.0273
    ρ_μ   = 0.9556  (annual: 0.8339)
    σ_μ   = 0.0048
    μ̄     = 0.0215  (8.6% ann.)
═══════════════════════════════════════════════════
  Simulating... done in 1.24s


MCResults(ModelParams(0.971, 0.4927, 0.0273, 0.000226, 0.9556, 0.0048, 0.0215, 0.99), 10000, 100, Dict{Int64, HorizonSnapshot}(5 => HorizonSnapshot(5, [-0.01854787807338984, 0.06580446054445327, 0.21586253651016246, 0.004138814339696692, 0.10927940543765781, 0.2714016984551785, 0.18595413282600415, -0.0653335545977277, 0.22210568507745582, 0.18274677150926336  …  -0.017794443632187296, 0.15606403331140706, -0.03932208186007974, 0.08669542230843565, 0.2924079874162809, 0.1413979399726021, -0.0009833427669024702, -0.09225878789410412, 0.18652257437136718, 0.12252448438252878], [0.0052215986712416925, 0.0048224834504817776, 0.0054615397255299986, 0.006507731938857355, 0.005166479039451572, 0.006224695032610112, 0.005481577907097433, 0.0030109819048470318, 0.006477878276607285, 0.0044883037822093085  …  0.007171634215393909, 0.004787835986448319, 0.0038376594284775826, 0.003937484539819026, 0.004528290987808482, 0.006590097565711323, 0.005097465153981641, 0.005391192961676752, 0.0023076480

## Summary Statistics

In [7]:
println("═══════════════════════════════════════════════════")
println("  Summary Statistics")
println("═══════════════════════════════════════════════════")

quantiles_list = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

for hy in sort(collect(keys(results.snapshots)))
    snap = results.snapshots[hy]
    println("\n── Horizon: $(hy) years ──")

    println("  Log Dividends (d):")
    println("    Mean:   $(round(mean(snap.d), digits=4))")
    println("    Std:    $(round(std(snap.d), digits=4))")

    println("  Trend Growth (g, quarterly %):")
    println("    Mean:   $(round(mean(snap.g)*100, digits=4))")
    println("    Std:    $(round(std(snap.g)*100, digits=4))")

    println("  Log Price-Dividend Ratio (pd):")
    println("    Mean:   $(round(mean(snap.pd), digits=4))")
    println("    Std:    $(round(std(snap.pd), digits=4))")

    println("  Annualised Log Return (%):")
    q_vals = quantile(snap.ann_log_r, quantiles_list)
    println("    Mean:   $(round(mean(snap.ann_log_r)*100, digits=2))%")
    println("    Std:    $(round(std(snap.ann_log_r)*100, digits=2))%")
    for (q, v) in zip(quantiles_list, q_vals)
        println("    Q$(round(Int, q*100)):    $(round(v*100, digits=2))%")
    end
end

═══════════════════════════════════════════════════
  Summary Statistics
═══════════════════════════════════════════════════

── Horizon: 1 years ──
  Log Dividends (d):
    Mean:   0.0201
    Std:    0.0734
  Trend Growth (g, quarterly %):
    Mean:   0.4999
    Std:    0.0445
  Log Price-Dividend Ratio (pd):
    Mean:   4.0956
    Std:    0.1766
  Annualised Log Return (%):
    Mean:   8.6%
    Std:    16.38%
    Q1:    -29.83%
    Q5:    -18.61%
    Q10:    -12.26%
    Q25:    -2.38%
    Q50:    8.52%
    Q75:    19.68%
    Q90:    29.65%
    Q95:    34.9%
    Q99:    47.25%

── Horizon: 5 years ──
  Log Dividends (d):
    Mean:   0.0992
    Std:    0.1417
  Trend Growth (g, quarterly %):
    Mean:   0.5007
    Std:    0.1014
  Log Price-Dividend Ratio (pd):
    Mean:   4.0982
    Std:    0.3023
  Annualised Log Return (%):
    Mean:   8.81%
    Std:    4.93%
    Q1:    -2.54%
    Q5:    0.7%
    Q10:    2.53%
    Q25:    5.51%
    Q50:    8.82%
    Q75:    12.05%
    Q90:    15.13%